In [ ]:
# @title Setup: Install scpn-quantum-control
!pip install -q scpn-quantum-control
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)


SPDX-License-Identifier: AGPL-3.0-or-later | Commercial license available
Â© Concepts 1996â€“2026 Miroslav Ĺ otek. All rights reserved.
Â© Code 2020â€“2026 Miroslav Ĺ otek. All rights reserved.
ORCID: 0009-0009-3560-0851
Contact: www.anulum.li | protoscience@anulum.li
scpn-quantum-control â€” Schumann Resonance and EEG Theta Band

Earth's electromagnetic cavity (ionosphere-surface) resonates at
7.83 Hz (fundamental), 14.3, 20.8, 27.3, 33.8 Hz (harmonics).
These fall EXACTLY in the EEG bands: theta, alpha, beta, gamma.
Coincidence? Or has the brain evolved to resonate with its
electromagnetic environment?

Tests:
1. Schumann frequencies vs EEG band centres
2. Statistical test: random match probability
3. Coupling mechanism: can Earth's field affect neural oscillators?
4. Schumann as external K_nm driver for circadian/ultradian rhythms
5. Planetary scale in the SCPN frequency hierarchy

In [ ]:
import json

In [ ]:
import numpy as np

In [ ]:
print("=" * 70)
print("SCHUMANN RESONANCE vs EEG BANDS")
print("=" * 70)

In [ ]:
# =====================================================================
# Schumann resonances (measured, Balser & Wagner 1960+)
# =====================================================================
schumann_modes = {
    "S1": 7.83,  # fundamental
    "S2": 14.3,
    "S3": 20.8,
    "S4": 27.3,
    "S5": 33.8,
    "S6": 39.0,  # weak but measurable
    "S7": 45.0,  # very weak
}

In [ ]:
# Theoretical: f_n = (c / 2*pi*R) * sqrt(n*(n+1))
# R = 6371 km, c = 3e8 m/s
R_earth = 6.371e6  # m
c_light = 3e8  # m/s

In [ ]:
print("Schumann resonances (measured vs theoretical):")
for n, (name, f_meas) in enumerate(schumann_modes.items(), 1):
    f_theory = (c_light / (2 * np.pi * R_earth)) * np.sqrt(n * (n + 1))
    print(f"  {name}: measured={f_meas:.2f} Hz, theory={f_theory:.2f} Hz")

In [ ]:
# =====================================================================
# EEG frequency bands
# =====================================================================
eeg_bands = {
    "delta": (1, 4, 2.5),
    "theta": (4, 8, 6.0),
    "alpha": (8, 13, 10.5),
    "beta_low": (13, 20, 16.5),
    "beta_high": (20, 30, 25.0),
    "gamma_low": (30, 45, 37.5),
    "gamma_high": (45, 100, 72.5),
}

In [ ]:
# TEST 1: Direct frequency comparison
print("\n" + "=" * 70)
print("TEST 1: SCHUMANN vs EEG BAND OVERLAP")
print("=" * 70)

In [ ]:
schumann_freqs = np.array(list(schumann_modes.values()))

In [ ]:
print(
    f"\n{'Schumann':>10s} {'Freq':>8s} {'Nearest EEG band':>20s} {'Band range':>15s} {'Inside?':>8s}"
)
print("-" * 65)
for name, freq in schumann_modes.items():
    best_band = None
    inside = False
    for band_name, (lo, hi, _centre) in eeg_bands.items():
        if lo <= freq <= hi:
            best_band = band_name
            inside = True
            break
    if best_band is None:
        # Find nearest
        distances = {
            bn: min(abs(freq - lo), abs(freq - hi)) for bn, (lo, hi, _) in eeg_bands.items()
        }
        best_band = min(distances, key=distances.get)

    lo, hi, _ = eeg_bands[best_band]
    print(
        f"{name:>10s} {freq:8.2f} {best_band:>20s} {lo:>5.0f}-{hi:<5.0f} Hz {'YES' if inside else 'no':>8s}"
    )

In [ ]:
# Count overlaps
n_inside = sum(
    1 for freq in schumann_freqs if any(lo <= freq <= hi for _, (lo, hi, _) in eeg_bands.items())
)
print(f"\n{n_inside}/{len(schumann_freqs)} Schumann modes fall inside EEG bands")

In [ ]:
# TEST 2: Statistical significance of overlap
print("\n" + "=" * 70)
print("TEST 2: PROBABILITY OF RANDOM MATCH")
print("=" * 70)

In [ ]:
# What's the probability that n_inside or more random frequencies
# in range [1, 50] Hz would fall inside EEG bands?
eeg_coverage = sum(hi - lo for _, (lo, hi, _) in eeg_bands.items())
total_range = 50.0  # Hz
p_single = eeg_coverage / total_range

In [ ]:
print(f"EEG bands cover {eeg_coverage:.0f} Hz out of 1-50 Hz range")
print(f"Probability of single random frequency in any EEG band: {p_single:.3f}")

In [ ]:
# Binomial test
from scipy.stats import binom

In [ ]:
p_random = 1 - binom.cdf(n_inside - 1, len(schumann_freqs), p_single)
print(f"P({n_inside}+ of {len(schumann_freqs)} random freqs in EEG bands): {p_random:.4f}")

In [ ]:
if p_random < 0.05:
    print("SIGNIFICANT: overlap is unlikely by chance")
else:
    print("NOT SIGNIFICANT: overlap could be chance (EEG bands are wide)")

In [ ]:
# TEST 3: Coupling strength estimate
print("\n" + "=" * 70)
print("TEST 3: CAN SCHUMANN FIELDS AFFECT NEURONS?")
print("=" * 70)

In [ ]:
# Schumann field amplitude: ~1 pT (picoTesla) magnetic, ~0.5 mV/m electric
B_schumann = 1e-12  # Tesla
E_schumann = 0.5e-3  # V/m

In [ ]:
# Neural threshold: ~100 uV for EEG detection
# Membrane potential: ~70 mV across ~7 nm
V_membrane = 70e-3  # V
d_membrane = 7e-9  # m
E_membrane = V_membrane / d_membrane  # V/m

In [ ]:
print(f"Schumann electric field: {E_schumann:.1e} V/m")
print(f"Membrane electric field: {E_membrane:.1e} V/m")
print(f"Ratio: {E_schumann / E_membrane:.2e}")
print(f"Schumann is {E_membrane / E_schumann:.0e}x weaker than membrane field")

In [ ]:
# But: stochastic resonance can amplify weak signals
# If the brain is near K_c, even tiny perturbations matter
# Signal-to-noise: does Schumann exceed thermal noise?
kB = 1.381e-23
T = 310  # K
# Thermal voltage noise: sqrt(4*kB*T*R*BW)
R_neuron = 1e8  # Ohm (typical membrane resistance)
BW = 10  # Hz (bandwidth of interest)
V_thermal = np.sqrt(4 * kB * T * R_neuron * BW)
print(f"\nThermal voltage noise (10 Hz BW): {V_thermal * 1e6:.2f} uV")

In [ ]:
# Schumann-induced voltage across typical neuron (10 um)
L_neuron = 10e-6  # m
V_schumann = E_schumann * L_neuron
print(f"Schumann voltage across neuron: {V_schumann * 1e6:.4f} uV")
print(f"Ratio to thermal: {V_schumann / V_thermal:.4f}")

In [ ]:
if V_schumann > V_thermal:
    print("Schumann EXCEEDS thermal noise -> detectable in principle")
else:
    print(f"Schumann is {V_thermal / V_schumann:.0f}x below thermal noise")
    print("Direct coupling unlikely. BUT:")
    print("  - Stochastic resonance could amplify")
    print("  - Coherent integration over many neurons could amplify")
    print("  - Resonant amplification at matching frequency")

In [ ]:
# TEST 4: Schumann harmonics as frequency template
print("\n" + "=" * 70)
print("TEST 4: SCHUMANN HARMONIC RATIOS vs EEG RATIOS")
print("=" * 70)

In [ ]:
# Are the Schumann harmonic ratios the same as EEG band ratios?
schumann_ratios = []
s_freqs = list(schumann_modes.values())
for i in range(len(s_freqs) - 1):
    ratio = s_freqs[i + 1] / s_freqs[i]
    schumann_ratios.append(ratio)
    print(f"  S{i + 1}/S{i + 2}: {ratio:.3f}")

In [ ]:
print(f"\nMean Schumann ratio: {np.mean(schumann_ratios):.3f}")

In [ ]:
# Compare to EEG ratios (from our neural oscillation notebook)
eeg_centres = [d[2] for d in eeg_bands.values()]
eeg_ratios = [eeg_centres[i + 1] / eeg_centres[i] for i in range(len(eeg_centres) - 1)]
print(f"Mean EEG ratio: {np.mean(eeg_ratios):.3f}")
print(f"Buzsaki ratio: ~{np.e:.3f}")

In [ ]:
# Schumann theoretical ratio: sqrt((n+1)(n+2)) / sqrt(n(n+1))
print("\nSchumann theoretical ratios: sqrt((n+1)(n+2)/n(n+1))")
for n in range(1, 7):
    r = np.sqrt((n + 1) * (n + 2) / (n * (n + 1)))
    print(f"  n={n}: {r:.4f}")
print("Schumann ratios DECREASE (approaching 1) â€” different from EEG")

In [ ]:
# TEST 5: Planetary oscillation hierarchy
print("\n" + "=" * 70)
print("TEST 5: PLANETARY OSCILLATION HIERARCHY")
print("=" * 70)

In [ ]:
# Earth has oscillations at many scales
planetary = {
    "Schumann_S1": 7.83,  # Hz
    "diurnal_tide": 1 / (12 * 3600),  # ~23 uHz (12-hr tide)
    "circadian": 1 / 86400,  # ~12 uHz
    "Chandler_wobble": 1 / (433 * 86400),  # ~27 nHz (433 days)
    "precession": 1 / (26000 * 365.25 * 86400),  # ~1.2 pHz
    "Milankovitch_41k": 1 / (41000 * 365.25 * 86400),
    "Milankovitch_100k": 1 / (100000 * 365.25 * 86400),
}

In [ ]:
print("Earth's oscillation hierarchy:")
sorted_planet = sorted(planetary.items(), key=lambda x: x[1], reverse=True)
for name, freq in sorted_planet:
    if freq > 1:
        print(f"  {name:25s}: {freq:.2f} Hz")
    elif freq > 1e-3:
        print(f"  {name:25s}: {freq * 1e3:.3f} mHz")
    elif freq > 1e-6:
        print(f"  {name:25s}: {freq * 1e6:.3f} uHz")
    elif freq > 1e-9:
        print(f"  {name:25s}: {freq * 1e9:.3f} nHz")
    else:
        print(f"  {name:25s}: {freq * 1e12:.3f} pHz")

In [ ]:
# Span
fmax = max(planetary.values())
fmin = min(planetary.values())
print(f"\nPlanetary span: {np.log10(fmax / fmin):.1f} decades")
print("Bio span (ETC to circadian): ~13 decades")
print("SCPN span: 1.2 decades")

In [ ]:
# Ratios between adjacent planetary oscillations
planet_freqs = [f for _, f in sorted_planet]
planet_ratios = [planet_freqs[i] / planet_freqs[i + 1] for i in range(len(planet_freqs) - 1)]
print("\nPlanetary frequency ratios (consecutive):")
for i, r in enumerate(planet_ratios):
    n1 = sorted_planet[i][0]
    n2 = sorted_planet[i + 1][0]
    print(f"  {n1:25s} / {n2:25s} = {r:.1e}")

In [ ]:
# SYNTHESIS
print("\n" + "=" * 70)
print("SYNTHESIS: SCHUMANN-EEG CONNECTION")
print("=" * 70)

In [ ]:
print(f"""
1. OVERLAP: {n_inside}/{len(schumann_freqs)} Schumann modes in EEG bands
   Statistical significance: p={p_random:.4f}
   {"SIGNIFICANT" if p_random < 0.05 else "Not significant (EEG bands are wide)"}

2. DIRECT COUPLING: Schumann field is {V_thermal / V_schumann:.0f}x below thermal noise
   Direct electromagnetic coupling to single neurons: UNLIKELY
   But: coherent integration, resonance, stochastic amplification possible

3. HARMONIC STRUCTURE: Schumann ratios DECREASE toward 1
   EEG ratios are roughly constant (~e)
   Different generating mechanisms (cavity vs neural network)

4. EVOLUTIONARY ARGUMENT:
   Life evolved in Earth's EM environment for 4 billion years.
   If Schumann provides a TIMING REFERENCE, brains that sync to it
   have an advantage (coordinated circadian, seasonal responses).
   The coupling need not be strong â€” just above stochastic resonance threshold.

5. The Schumann-theta coincidence (7.83 Hz in theta band) may be:
   a) Pure coincidence (EEG bands are wide)
   b) Evolutionary tuning (brains adapted to resonate)
   c) Physics constraint (similar cavity/network size â†’ similar frequency)
   We cannot distinguish these from our data.

HONEST ASSESSMENT: The frequency overlap is suggestive but the coupling
mechanism is 10^7x too weak for direct influence. Any real effect would
require amplification via stochastic resonance or coherent summation
across neural populations. This is a hypothesis, not a finding.
""")

In [ ]:
results = {
    "n_schumann_in_eeg": n_inside,
    "n_schumann_total": len(schumann_freqs),
    "overlap_p_value": round(float(p_random), 4),
    "schumann_to_thermal_ratio": round(float(V_schumann / V_thermal), 6),
    "coupling_gap_orders": round(float(np.log10(V_thermal / V_schumann)), 1),
    "schumann_mean_ratio": round(float(np.mean(schumann_ratios)), 3),
    "eeg_mean_ratio": round(float(np.mean(eeg_ratios)), 3),
}

In [ ]:
print("\n--- JSON RESULTS ---")
print(json.dumps(results, indent=2))